# Reto 1: Consulta y análisis de datos meteorológicos

## 12. Amplía la investigación

Ampliaremos el análisis de tres formas: compararemos varias ciudades, estudiaremos dos años diferentes y añadiremos la precipitación diaria. Así comprobaremos cómo se puede reutilizar el mismo programa cambiando las coordenadas, el periodo o las variables solicitadas.

## Preparación

La función `obtener_datos()` permite realizar distintas consultas a Open-Meteo sin repetir todo el código. Recibe las coordenadas, las fechas y las variables meteorológicas que necesita cada apartado.

In [ ]:
from pathlib import Path
from statistics import mean

import matplotlib.pyplot as plt
import requests

url = "https://archive-api.open-meteo.com/v1/archive"

def obtener_datos(latitud, longitud, fecha_inicio, fecha_fin, variables):
    parametros = {
        "latitude": latitud,
        "longitude": longitud,
        "start_date": fecha_inicio,
        "end_date": fecha_fin,
        "daily": ",".join(variables),
        "timezone": "Europe/Madrid",
    }
    respuesta = requests.get(url, params=parametros, timeout=30)
    respuesta.raise_for_status()
    return respuesta.json()["daily"]

variables_temperatura = [
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
]

directorio_salida = Path("Reto1/graficos") if Path("Reto1").exists() else Path("graficos")
directorio_salida.mkdir(parents=True, exist_ok=True)
print("Gráficos guardados en:", directorio_salida.resolve())

## 1. Compara Jaén, Madrid y Sevilla

Consultamos enero de 2025 para las tres ciudades. Los resultados se organizan en un diccionario cuya clave es el nombre de la ciudad. Para comparar periodos completos calculamos el promedio de las temperaturas medias, máximas y mínimas diarias.

In [ ]:
coordenadas_ciudades = {
    "Jaén": {"latitud": 37.7796, "longitud": -3.7849},
    "Madrid": {"latitud": 40.4168, "longitud": -3.7038},
    "Sevilla": {"latitud": 37.3891, "longitud": -5.9845},
}

datos_ciudades = {}

for ciudad, coordenadas in coordenadas_ciudades.items():
    datos_ciudades[ciudad] = obtener_datos(
        coordenadas["latitud"],
        coordenadas["longitud"],
        "2025-01-01",
        "2025-01-31",
        variables_temperatura,
    )

resumen_ciudades = {}

for ciudad, diarios in datos_ciudades.items():
    resumen_ciudades[ciudad] = {
        "media": round(mean(diarios["temperature_2m_mean"]), 2),
        "media_maximas": round(mean(diarios["temperature_2m_max"]), 2),
        "media_minimas": round(mean(diarios["temperature_2m_min"]), 2),
        "maxima_absoluta": max(diarios["temperature_2m_max"]),
        "minima_absoluta": min(diarios["temperature_2m_min"]),
    }

for ciudad, resumen in resumen_ciudades.items():
    print(ciudad, "->", resumen)

### Resultados de la comparación

| Ciudad | Media de máximas | Temperatura media | Media de mínimas | Máxima absoluta | Mínima absoluta |
|---|---:|---:|---:|---:|---:|
| Jaén | 13.34 °C | 8.82 °C | 4.67 °C | 18.6 °C | -1.1 °C |
| Madrid | 11.01 °C | 6.61 °C | 3.01 °C | 17.8 °C | -3.9 °C |
| Sevilla | 15.84 °C | 11.22 °C | 7.52 °C | 18.9 °C | 0.7 °C |

Sevilla fue la ciudad más cálida del periodo en las tres medidas promedio. Madrid fue la más fría y registró también la mínima absoluta más baja. Jaén quedó en una posición intermedia.

In [ ]:
ciudades = list(resumen_ciudades.keys())
posiciones = list(range(len(ciudades)))
ancho = 0.25

fig, ax = plt.subplots(figsize=(9, 6))
ax.bar(
    [posicion - ancho for posicion in posiciones],
    [resumen_ciudades[ciudad]["media_maximas"] for ciudad in ciudades],
    ancho,
    color="tab:red",
    label="Media de máximas",
)
ax.bar(
    posiciones,
    [resumen_ciudades[ciudad]["media"] for ciudad in ciudades],
    ancho,
    color="tab:orange",
    label="Temperatura media",
)
ax.bar(
    [posicion + ancho for posicion in posiciones],
    [resumen_ciudades[ciudad]["media_minimas"] for ciudad in ciudades],
    ancho,
    color="tab:blue",
    label="Media de mínimas",
)

ax.set_title("Comparación de temperaturas — enero de 2025")
ax.set_xlabel("Ciudad")
ax.set_ylabel("Temperatura (°C)")
ax.set_xticks(posiciones, ciudades)
ax.grid(axis="y", alpha=0.3)
ax.legend()
fig.tight_layout()

ruta_ciudades = directorio_salida / "12_comparacion_ciudades.png"
fig.savefig(ruta_ciudades, dpi=150, bbox_inches="tight")
plt.show()
print("Guardado en:", ruta_ciudades)

## 2. Compara enero de 2024 con enero de 2025

Comparamos en Jaén las medias del periodo, las medias de las máximas y mínimas, y los valores extremos de ambos meses. Los datos de 2025 ya se obtuvieron en el apartado anterior, por lo que solo es necesario realizar una nueva petición para 2024.

In [ ]:
datos_2024 = obtener_datos(
    37.7796,
    -3.7849,
    "2024-01-01",
    "2024-01-31",
    variables_temperatura,
)

datos_por_ano = {
    "2024": datos_2024,
    "2025": datos_ciudades["Jaén"],
}

resumen_anual = {}

for ano, diarios in datos_por_ano.items():
    resumen_anual[ano] = {
        "media": round(mean(diarios["temperature_2m_mean"]), 2),
        "media_maximas": round(mean(diarios["temperature_2m_max"]), 2),
        "media_minimas": round(mean(diarios["temperature_2m_min"]), 2),
        "maxima_absoluta": max(diarios["temperature_2m_max"]),
        "minima_absoluta": min(diarios["temperature_2m_min"]),
    }

for ano, resumen in resumen_anual.items():
    print(ano, "->", resumen)

diferencia_medias = round(resumen_anual["2025"]["media"] - resumen_anual["2024"]["media"], 2)
print("Diferencia entre las medias de 2025 y 2024:", diferencia_medias, "°C")

### Resultados de la comparación anual

| Enero | Media de máximas | Temperatura media | Media de mínimas | Máxima absoluta | Mínima absoluta |
|---|---:|---:|---:|---:|---:|
| 2024 | 15.63 °C | 10.44 °C | 5.86 °C | 21.6 °C | 2.1 °C |
| 2025 | 13.34 °C | 8.82 °C | 4.67 °C | 18.6 °C | -1.1 °C |

Enero de 2025 tuvo una temperatura media `1.62 °C` inferior a enero de 2024. También presentó menores máximas y mínimas, y llegó a registrar temperaturas bajo cero. Esto permite describir la diferencia entre estos dos meses concretos, pero no demuestra por sí solo una tendencia climática.

In [ ]:
anos = list(resumen_anual.keys())
posiciones = list(range(len(anos)))
ancho = 0.25

fig, ax = plt.subplots(figsize=(8, 6))
ax.bar(
    [posicion - ancho for posicion in posiciones],
    [resumen_anual[ano]["media_maximas"] for ano in anos],
    ancho,
    color="tab:red",
    label="Media de máximas",
)
ax.bar(
    posiciones,
    [resumen_anual[ano]["media"] for ano in anos],
    ancho,
    color="tab:orange",
    label="Temperatura media",
)
ax.bar(
    [posicion + ancho for posicion in posiciones],
    [resumen_anual[ano]["media_minimas"] for ano in anos],
    ancho,
    color="tab:blue",
    label="Media de mínimas",
)

ax.set_title("Jaén: enero de 2024 frente a enero de 2025")
ax.set_xlabel("Año")
ax.set_ylabel("Temperatura (°C)")
ax.set_xticks(posiciones, anos)
ax.grid(axis="y", alpha=0.3)
ax.legend()
fig.tight_layout()

ruta_anual = directorio_salida / "12_comparacion_2024_2025.png"
fig.savefig(ruta_anual, dpi=150, bbox_inches="tight")
plt.show()
print("Guardado en:", ruta_anual)

## 3. Analiza la precipitación diaria

Solicitamos `precipitation_sum`, que representa la precipitación diaria acumulada en milímetros. Calcularemos el total del mes, la media diaria, el porcentaje de días con precipitación y el día más lluvioso.

In [ ]:
datos_precipitacion = obtener_datos(
    37.7796,
    -3.7849,
    "2025-01-01",
    "2025-01-31",
    ["precipitation_sum"],
)

fechas_precipitacion = datos_precipitacion["time"]
precipitaciones = datos_precipitacion["precipitation_sum"]

precipitacion_total = round(sum(precipitaciones), 1)
precipitacion_media = round(mean(precipitaciones), 2)
dias_con_precipitacion = sum(valor > 0 for valor in precipitaciones)
porcentaje_dias_precipitacion = round(dias_con_precipitacion / len(precipitaciones) * 100, 2)
precipitacion_maxima = max(precipitaciones)
indice_dia_mas_lluvioso = precipitaciones.index(precipitacion_maxima)
fecha_mas_lluviosa = fechas_precipitacion[indice_dia_mas_lluvioso]

print("Precipitación total:", precipitacion_total, "mm")
print("Precipitación media diaria:", precipitacion_media, "mm")
print("Días con precipitación:", dias_con_precipitacion)
print("Porcentaje de días con precipitación:", porcentaje_dias_precipitacion, "%")
print("Día más lluvioso:", fecha_mas_lluviosa)
print("Precipitación máxima diaria:", precipitacion_maxima, "mm")

### Resultados de precipitación

- La precipitación total fue de `60.2 mm`.
- La media diaria fue de `1.94 mm`.
- Se registró precipitación durante 14 de los 31 días, un `45.16 %` del mes.
- El día más lluvioso fue el `2025-01-31`, con `17.7 mm`.

La media diaria incluye también los días sin lluvia. Por ello no representa la cantidad media de los días lluviosos, sino la precipitación total repartida entre todos los días del mes.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(fechas_precipitacion, precipitaciones, color="tab:blue")
ax.axhline(
    precipitacion_media,
    color="black",
    linestyle="--",
    label=f"Media diaria: {precipitacion_media:.2f} mm",
)

ax.set_title("Precipitación diaria — Jaén, enero de 2025")
ax.set_xlabel("Fecha")
ax.set_ylabel("Precipitación (mm)")
ax.set_xticks(range(0, len(fechas_precipitacion), 3), fechas_precipitacion[::3], rotation=45, ha="right")
ax.grid(axis="y", alpha=0.3)
ax.legend()
fig.tight_layout()

ruta_precipitacion = directorio_salida / "12_precipitacion_diaria.png"
fig.savefig(ruta_precipitacion, dpi=150, bbox_inches="tight")
plt.show()
print("Guardado en:", ruta_precipitacion)

## Comprueba tu comprensión

### 1. ¿Por qué se cambian las coordenadas para comparar ciudades?

**Respuesta:**

La latitud y la longitud identifican la ubicación de la consulta. Mantener las mismas fechas y variables, pero cambiar las coordenadas, permite obtener datos equivalentes de cada ciudad y realizar una comparación coherente.

### 2. ¿Por qué dos meses no permiten establecer una tendencia climática?

**Respuesta:**

El tiempo meteorológico varía de un año a otro. Comparar enero de 2024 y enero de 2025 permite describir esos dos meses, pero una tendencia climática necesita series de muchos años para separar una evolución persistente de la variabilidad natural. También habría que considerar la calidad y homogeneidad de los datos.

### 3. ¿Cómo se identifica un día con precipitación?

**Respuesta:**

En este ejercicio se considera que hubo precipitación cuando `precipitation_sum > 0`. El porcentaje se calcula dividiendo esos días entre el número total de días y multiplicando por 100. En un estudio meteorológico más preciso podría utilizarse un umbral mínimo para evitar cantidades inapreciables.

## Conclusiones

1. Sevilla presentó las temperaturas promedio más altas de las tres ciudades y Madrid las más bajas; Jaén ocupó una posición intermedia.
2. Enero de 2025 fue más frío que enero de 2024 en Jaén: su media fue `1.62 °C` inferior. Esta diferencia solo describe los dos meses estudiados y no demuestra una tendencia climática.
3. En Jaén se acumularon `60.2 mm` durante enero de 2025. El 31 de enero concentró `17.7 mm`, casi el 30 % del total mensual, lo que muestra que la lluvia no se distribuyó uniformemente.

El análisis demuestra que una misma API y una función reutilizable permiten ampliar la investigación modificando la ubicación, el periodo y las variables. Sus principales limitaciones son la brevedad de los periodos comparados y que cada ciudad se representa mediante un único punto de la cuadrícula meteorológica.